# Import libraries

In [67]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

# Load full ground truth

In [68]:
def load_gt(gt_fname):
    with open(gt_fname) as f:
        gt = json.load(f)

    rows_gt = []
    for video_id, info in gt["database"].items():
        duration = info["duration"]
        subset = info["subset"]
        for s in info["annotations"]:
            rows_gt.append(
                {
                    "video_id": video_id,
                    "duration": duration,
                    "subset": subset,
                    "start": s["segment"][0],
                    "end": s["segment"][1],
                    "label": s["label"],
                }
            )

    return pd.DataFrame(rows_gt)

In [69]:
gt_df = load_gt("../data/charades/annotations/charades.json")

# Load EXO_EGO annotations

In [70]:
training_ego_df = pd.read_csv("../data/charades/annotations/CharadesEgo_v1_train_only1st.csv")
testing_ego_df  = pd.read_csv("../data/charades/annotations/CharadesEgo_v1_test_only1st.csv")
# prepare to stack training and testing splits
training_ego_df = training_ego_df[['id', 'charades_video']]
testing_ego_df = testing_ego_df[['id', 'charades_video']]
ego_df = pd.concat([training_ego_df, testing_ego_df], axis=0)
# clean ego_df: remove rows with na value
ego_df = ego_df.dropna()
ego_df.head()

,id,charades_video
0,D3TR8EGO,1K0SU
3,6U5TEEGO,YFZRG
4,KCSBQEGO,SKUOZ
5,F6XROEGO,V7T91
7,28DG6EGO,52MV9


# Sample the 10 most frequent classes (~ 200 segments for each class)

In [71]:
# Create sample classes and save them to a file
most_sample_classes = sorted(
    gt_df["label"].value_counts()[:10].index.tolist()
)  # Select the 10 most frequent classes

# Step 0: Filter the original dataset to include only the most frequent classes
most_gt_df = gt_df[gt_df["label"].isin(most_sample_classes)].copy()

# Step 1: Sample ~200 rows per label from the filtered dataset
most_sample_rows = most_gt_df.groupby('label', group_keys=False).sample(n=200, random_state=42).copy()

# Step 2: Get all unique video_ids from those sampled rows
video_ids_to_include = most_sample_rows["video_id"].unique()

# Step 3: Filter the most gt df to include all rows with those video_ids
most_sample_gt_df = most_gt_df[most_gt_df["video_id"].isin(video_ids_to_include)].copy()

In [72]:
most_sample_gt_df.head()

,video_id,duration,subset,start,end,label
14,J3UGZ,23.00,training,15.9,20.9,Someone is going from standing to sitting
15,J3UGZ,23.00,training,21.1,24.0,Someone is standing up from somewhere
16,J3UGZ,23.00,training,21.1,24.0,Someone is smiling
17,J3UGZ,23.00,training,21.1,24.0,Holding a phone/camera
29,0F1VF,29.71,training,6.0,10.9,Someone is going from standing to sitting


# Enrich information of most_sample_gt_df to get relevant ego_id

In [73]:
df_exo_ego = most_sample_gt_df.merge(ego_df, left_on='video_id', right_on='charades_video', how='inner').rename(columns={'id': 'ego_id'})
df_exo_ego.head()

,video_id,duration,subset,start,end,label,ego_id,charades_video
0,0F1VF,29.71,training,6.0,10.9,Someone is going from standing to sitting,R959XEGO,0F1VF
1,0F1VF,29.71,training,15.7,20.9,Someone is standing up from somewhere,R959XEGO,0F1VF
2,WN2N7,24.83,training,0.7,7.2,Holding a phone/camera,H7PVHEGO,WN2N7
3,F1DQD,31.92,training,0.0,33.0,Someone is smiling,G2AOYEGO,F1DQD
4,F1DQD,31.92,training,0.0,33.0,Sitting in a chair,G2AOYEGO,F1DQD


In [78]:
def build_exo_ego_dataset(
    sample_gt_df,
    category_idx_fname="../data/charades/annotations/sample_category_idx.txt",
    gt_fname="../data/charades/annotations/sample_charades.json",
    haveExo=True,
    haveEgo=True,
):
    # Save category index to a text file
    actions = sorted(sample_gt_df["label"].unique().tolist())
    with open(category_idx_fname, "w") as f:
        for action in actions:
            f.write(action + "\n")

    # Save sample annotations to JSON file
    output = {"version": "Most_Exo_Ego", "database": {}}

    for video_id, group in sample_gt_df.groupby("video_id"):
        # keep duration/subset from the original gt
        duration = sample_gt_df[sample_gt_df["video_id"] == video_id]["duration"].iloc[0]
        subset = sample_gt_df[sample_gt_df["video_id"] == video_id]["subset"].iloc[0]
        ego_id = sample_gt_df[sample_gt_df["video_id"] == video_id]["ego_id"].iloc[0]
        exo_id = ego_id[:-3]

        annotations = []
        for _, row in group.iterrows():
            annotations.append(
                {"segment": [row["start"], row["end"]], "label": row["label"]}
            )

        if haveEgo:
            # for ego view extraction
            output["database"][ego_id] = {
                "video_id": video_id,  # not sure if necessary
                "duration": duration,
                "subset": subset,
                "annotations": annotations,
            }

        if haveExo:
            # for exo view extraction
            output["database"][exo_id] = {
                "video_id": video_id,  # not sure if necessary
                "duration": duration,
                "subset": subset,
                "annotations": annotations,
            }

    with open(gt_fname, "w") as f:
        json.dump(output, f, indent=2)

In [79]:
build_exo_ego_dataset(
    df_exo_ego,
    category_idx_fname="../data/charades/annotations/exo_only_category_idx.txt",
    gt_fname="../data/charades/annotations/exo_only_charades.json",
    haveEgo=False,
    haveExo=True,
)